# PROYEK DATA SCIENCE: PREDIKSI HARGA PENAWARAN RUMAH TINGGAL KOTA DEPOK
### Proyek Akhir Bengkel Koding (EarlyBird Bengkod)
---

## 1. Business Understanding & Problem Framing

### 1.1 Latar Belakang Masalah
Kota Depok merupakan salah satu kota penyangga utama DKI Jakarta (*hinterland*) dengan dinamika pasar properti yang sangat aktif. Pertumbuhan kawasan hunian di Depok sangat bervariasi: mulai dari kawasan *established/premium* yang berbatasan langsung dengan Jakarta Selatan (seperti Cinere dan Pangkalan Jati), kawasan pendidikan (Beji - Universitas Indonesia), hingga kawasan ekspansi perumahan baru (*sunrise area* seperti Sawangan dan Cilodong).

Bagi calon pembeli rumah pertama (*first-time homebuyers*), investor, maupun agen properti, menentukan apakah suatu penawaran harga rumah wajar (*fair market value*) atau terlalu mahal (*overpriced*) merupakan tantangan besar karena:
1. Harga listing di portal properti sering kali tidak terstandardisasi dan mengandung banyak *noise*.
2. Nilai properti sangat dipengaruhi oleh kombinasi spesifikasi fisik (Luas Tanah, Luas Bangunan, Kamar Tidur, Kamar Mandi, Garasi) serta faktor lokasi (Kecamatan).

### 1.2 Tujuan Proyek (*Project Goals*)
1. Membangun model machine learning berbasis regresi yang mampu memprediksi estimasi harga pasar wajar rumah di Kota Depok berdasarkan karakteristik fisik dan wilayahnya.
2. Memperbaiki kualitas data kotor (*raw scraped data*) melalui *regex data recovery* dan penanganan outlier ekstrem secara objektif.
3. Mengembangkan pipeline pemodelan terintegrasi (*end-to-end*) yang bebas dari *data leakage*, menggunakan transformasi target logaritmik, standarisasi fitur, serta evaluasi metrik ganda (MAE, MAPE, dan $R^2$).
4. Menyediakan model yang siap di-*deploy* ke aplikasi web interaktif berbasis Streamlit.


## 2. Data Loading & Exploratory Data Analysis (EDA)

Pada tahap ini, kita memuat dataset `Raw-Final-Rumah.csv`, memeriksa integritas kolom, tipe data, mendeteksi nilai kosong (*missing values*), karakter mencurigakan (*suspect values*), serta mengeksplorasi distribusi harga dan karakteristik wilayah.

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import kruskal
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.inspection import permutation_importance
import joblib

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: "%.2f" % x)

print("Semua pustaka berhasil dimuat.")

In [ ]:
# Memuat dataset mentah
df_raw = pd.read_csv("Raw-Final-Rumah.csv")

print("Dimensi dataset:", df_raw.shape)
display(df_raw.head())
df_raw.info()

### 2.1 Pemeriksaan Missing Value & Karakter Mencurigakan (Suspects)
Mengadopsi pendekatan dari silabus Bengkel Koding, kita memeriksa tidak hanya nilai `null`/`NaN` standar, tetapi juga karakter *placeholder* seperti `?`, `-`, `unknown`, dsb.

In [ ]:
# Cek missing values standar
print("Jumlah Missing Values per Kolom:")
display(df_raw.isnull().sum().to_frame("Jumlah Missing"))

# Cek nilai-nilai mencurigakan (suspects)
suspects = ["?", "??", "NA", "N/A", "na", "None", "null", "Null", "Unknown", "UNK", "-", "--", " ", "Nan", "XXX"]
suspect_summary = {val: (df_raw == val).sum().sum() for val in suspects}
print("\nRingkasan Nilai Suspect:", {k: v for k, v in suspect_summary.items() if v > 0})

print("\nJumlah Duplikat Persis:", df_raw.duplicated().sum())

### 2.2 Fungsi Parsing Awal untuk Eksplorasi
Sebelum menganalisis korelasi dan distribusi harga, kita mengonversi teks harga (misal `"2,5 Miliar"`, `"385 Juta"`) ke nilai nominal numerik Rupiah.

In [ ]:
def parse_harga(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower().replace("\u00a0", " ")
    match = re.search(r"(\d[\d.,]*)\s*(miliar|milyar|juta|jt)\b", text)
    if not match:
        return np.nan
    angka_text, satuan = match.group(1), match.group(2)
    if "," in angka_text:
        angka_text = angka_text.replace(".", "").replace(",", ".")
    elif angka_text.count(".") > 1:
        angka_text = angka_text.replace(".", "")
    angka = float(angka_text)
    if satuan in ["miliar", "milyar"]:
        return angka * 1_000_000_000
    return angka * 1_000_000

df_eda = df_raw.copy()
df_eda["Harga_Rp"] = df_eda["Harga"].apply(parse_harga)

print("Ringkasan Statistik Harga Kasar (Sebelum Pembersihan):")
display(df_eda["Harga_Rp"].describe().to_frame())

### 2.3 Analisis Distribusi Target: Normal vs Skala Logaritmik
Perhatikan perbedaan bentuk sebaran antara harga asli dalam Rupiah mentah dengan harga yang ditransformasikan dengan $\log_{10}$.

In [ ]:
harga_valid = df_eda["Harga_Rp"].dropna()
harga_positif = harga_valid[harga_valid > 0]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot distribusi asli
sns.histplot(harga_positif, bins=40, kde=True, ax=axes[0], color="#2563eb")
axes[0].set_title("Distribusi Harga Asli (Sangat Skewed)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Harga (Rp)")

# Plot distribusi log10
sns.histplot(np.log10(harga_positif), bins=40, kde=True, ax=axes[1], color="#059669")
axes[1].set_title("Distribusi Log10 Harga (Mendekati Normal/Gaussian)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Log10 Harga (Rp)")

plt.tight_layout()
plt.show()

### 2.4 Analisis Median Harga per Kecamatan di Kota Depok
Distribusi median harga properti per kecamatan memperlihatkan disparitas nilai tanah yang signifikan antar wilayah di Depok.

In [ ]:
ringkasan_kecamatan = (
    df_eda.dropna(subset=["Harga_Rp"])
    .groupby("Kecamatan")["Harga_Rp"]
    .agg(Jumlah_Listing="count", Median="median", Rata_Rata="mean", Min="min", Max="max")
    .sort_values("Median", ascending=False)
)
display(ringkasan_kecamatan)

plt.figure(figsize=(10, 6))
ringkasan_kecamatan["Median"].sort_values().plot(kind="barh", color="#3b82f6")
plt.title("Median Harga Properti Berdasarkan Kecamatan (Rp)", fontsize=14, fontweight="bold")
plt.xlabel("Median Harga (Rp)")
plt.ylabel("Kecamatan")
plt.tight_layout()
plt.show()

## 3. Data Cleaning, Regex Data Recovery & Outlier Handling

Salah satu keunikan pada dataset properti ini adalah terjadinya **anomali penempatan kolom** pada saat *web scraping*. Seringkali spesifikasi Luas Tanah (`LT : ... m²`) atau Luas Bangunan (`LB : ... m²`) masuk ke dalam kolom `Kamar Tidur`, `Kamar Mandi`, atau `Garasi`.

Pada bagian ini, kita menerapkan:
1. **Regex Data Recovery**: Memindai seluruh kolom untuk mengembalikan nilai Luas Tanah (LT) dan Luas Bangunan (LB) yang tersesat.
2. **Numeric Parsing**: Membersihkan kolom kamar tidur, kamar mandi, dan garasi dari teks.
3. **Outlier Filtering (Adopsi Prinsip Bengkod_DS_All)**: Menghapus listing sewa/DP (< Rp 150 Juta), anomali harga (> Rp 20 Miliar), dan dimensi tanah/bangunan di luar batas akal sehat perumahan residensial.

In [ ]:
def parse_angka(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if re.fullmatch(r"\d+(?:[.,]\d+)?", text):
        return float(text.replace(",", "."))
    return np.nan

def parse_area_label(value):
    if pd.isna(value):
        return None, np.nan
    text = str(value).strip().lower()
    match = re.search(r"\b(lt|lb)\b\s*:?\s*(\d+(?:[.,]\d+)?)\s*m(?:²|2)?\b", text)
    if not match:
        return None, np.nan
    return match.group(1).upper(), float(match.group(2).replace(",", "."))

def repair_area_row(row):
    sumber = ["Kamar Tidur", "Kamar Mandi", "Garasi", "Luas Tanah", "Luas Bangunan"]
    kandidat_lt, kandidat_lb = [], []
    for col in sumber:
        label, nilai = parse_area_label(row[col])
        if label == "LT":
            kandidat_lt.append(nilai)
        elif label == "LB":
            kandidat_lb.append(nilai)
    
    def pilih_nilai(kandidat):
        nilai_unik = sorted(set(kandidat))
        if len(nilai_unik) == 1:
            return nilai_unik[0], False
        if len(nilai_unik) == 0:
            return np.nan, False
        return np.nan, True
        
    lt, _ = pilih_nilai(kandidat_lt)
    lb, _ = pilih_nilai(kandidat_lb)
    return pd.Series({"LT_recovered": lt, "LB_recovered": lb})

print("Menjalankan Regex Data Recovery pada baris data...")
hasil_repair = df_raw.apply(repair_area_row, axis=1)

df_clean = df_raw.copy()
df_clean["Harga_Rp"] = df_clean["Harga"].apply(parse_harga)
for col in ["Kamar Tidur", "Kamar Mandi", "Garasi"]:
    df_clean[col + "_num"] = df_clean[col].apply(parse_angka)

df_clean["Luas Tanah_num"] = hasil_repair["LT_recovered"]
df_clean["Luas Bangunan_num"] = hasil_repair["LB_recovered"]

print("Luas Tanah berhasil dipulihkan :", df_clean["Luas Tanah_num"].notna().sum())
print("Luas Bangunan berhasil dipulihkan:", df_clean["Luas Bangunan_num"].notna().sum())

### 3.1 Eliminasi Outlier Ekstrem & Anomali
Berikut adalah filter domain properti untuk membersihkan listing cacat:
- **Harga Properti**: Rp 150 Juta $\le$ `Harga_Rp` $\le$ Rp 20 Miliar (membuang listing sewa kamar/DP bulanan dan properti super-komersial).
- **Luas Tanah**: 25 $m^2$ $\le$ `Luas Tanah_num` $\le$ 1.200 $m^2$.
- **Luas Bangunan**: 20 $m^2$ $\le$ `Luas Bangunan_num` $\le$ 800 $m^2$.
- **Jumlah Kamar Tidur & Mandi**: 1 s.d. 12 kamar.

In [ ]:
# Filter Domain Properti Wajar
mask_harga = (df_clean["Harga_Rp"] >= 150_000_000) & (df_clean["Harga_Rp"] <= 20_000_000_000)
mask_lt = df_clean["Luas Tanah_num"].isna() | ((df_clean["Luas Tanah_num"] >= 25) & (df_clean["Luas Tanah_num"] <= 1200))
mask_lb = df_clean["Luas Bangunan_num"].isna() | ((df_clean["Luas Bangunan_num"] >= 20) & (df_clean["Luas Bangunan_num"] <= 800))
mask_kt = df_clean["Kamar Tidur_num"].isna() | ((df_clean["Kamar Tidur_num"] >= 1) & (df_clean["Kamar Tidur_num"] <= 12))
mask_km = df_clean["Kamar Mandi_num"].isna() | ((df_clean["Kamar Mandi_num"] >= 1) & (df_clean["Kamar Mandi_num"] <= 10))

df_filtered = df_clean[mask_harga & mask_lt & mask_lb & mask_kt & mask_km].drop_duplicates().copy()

print(f"Data awal sebelum filtering : {df_raw.shape[0]} baris")
print(f"Data bersih setelah filtering: {df_filtered.shape[0]} baris")
print(f"Total baris outlier/duplikat dibuang: {df_raw.shape[0] - df_filtered.shape[0]} baris")

kolom_stat = ["Harga_Rp", "Kamar Tidur_num", "Kamar Mandi_num", "Garasi_num", "Luas Tanah_num", "Luas Bangunan_num"]
display(df_filtered[kolom_stat].describe().T)

## 4. Data Splitting (Train-Test Split Bersih)

Untuk menjaga validitas evaluasi dan mencegah kebocoran data (*data leakage*), pemisahan data training (80%) dan data testing (20%) dilakukan langsung pada dataset yang telah bersih menggunakan *random state* yang terkontrol.

In [ ]:
fitur = [
    "Kamar Tidur_num",
    "Kamar Mandi_num",
    "Garasi_num",
    "Luas Tanah_num",
    "Luas Bangunan_num",
    "Kecamatan"
]

X = df_filtered[fitur].copy()
y = df_filtered["Harga_Rp"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print(f"Dimensi Data Training (X_train): {X_train.shape}")
print(f"Dimensi Data Testing  (X_test) : {X_test.shape}")

## 5. Preprocessing & Pipeline Architecture

Menggabungkan keunggulan arsitektur Scikit-Learn `Pipeline` dari `EarlyBird` dengan standarisasi `StandardScaler` dari `Bengkod_DS_All`:
1. **Fitur Numerik**: Diimputasi nilai hilang dengan median, kemudian distandarisasi (*mean = 0, std = 1*).
2. **Fitur Kategorikal (`Kecamatan`)**: Diimputasi dengan nilai konstanta `"Unknown"`, kemudian di-encode dengan `OneHotEncoder(handle_unknown='ignore')`.
3. **Transformasi Target Logaritmik**: Menggunakan `TransformedTargetRegressor(func=np.log1p, inverse_func=np.expm1)` sehingga model belajar pada skala log yang stabil tanpa terdistorsi harga ekstrem, namun otomatis mengembalikan estimasi dalam satuan Rupiah pada fungsi `.predict()`.

In [ ]:
kolom_numerik = ["Kamar Tidur_num", "Kamar Mandi_num", "Garasi_num", "Luas Tanah_num", "Luas Bangunan_num"]
kolom_kategori = ["Kecamatan"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            kolom_numerik
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            kolom_kategori
        )
    ]
)

print("Preprocessor Pipeline berhasil dikonfigurasi:")
display(preprocessor)

## 6. Model Benchmarking

Kita membandingkan 5 algoritma regresi dengan pipeline preprocessing dan transformasi target yang identik:
1. **Ridge Regression** (Model Linier dengan regularisasi L2)
2. **K-Nearest Neighbors (KNN)** (Model berbasis jarak)
3. **Decision Tree Regressor** (Model pohon keputusan non-linier)
4. **Random Forest Regressor** (Ensemble Bagging)
5. **Gradient Boosting Regressor** (Ensemble Boosting)

Metrik yang dihitung:
- **MAE (Rp)**: *Mean Absolute Error* dalam satuan Rupiah.
- **MAPE (%)**: *Mean Absolute Percentage Error*.
- **R² Score**: Koefisien Determinasi (persentase variansi target yang dapat dijelaskan model).

In [ ]:
def evaluasi_model(y_true, y_pred):
    mask = np.asarray(y_true) > 0
    mape = mean_absolute_percentage_error(np.asarray(y_true)[mask], np.asarray(y_pred)[mask]) * 100
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {"MAE (Rp)": mae, "MAPE (%)": mape, "R2": r2}

candidate_models = {
    "Ridge Regression": Ridge(alpha=1.0),
    "K-Nearest Neighbors": KNeighborsRegressor(n_neighbors=7),
    "Decision Tree": DecisionTreeRegressor(max_depth=12, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42)
}

hasil_benchmark = []
fitted_pipelines = {}

print("Memulai proses pelatihan dan benchmarking model...")
for nama, model in candidate_models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", TransformedTargetRegressor(
            regressor=model,
            func=np.log1p,
            inverse_func=np.expm1
        ))
    ])
    
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    metrik = evaluasi_model(y_test, y_pred)
    metrik["Model"] = nama
    hasil_benchmark.append(metrik)
    fitted_pipelines[nama] = pipe
    print(f"Selesai: {nama:20s} | MAE: Rp {metrik['MAE (Rp)']:12,.0f} | MAPE: {metrik['MAPE (%)']:5.2f}% | R2: {metrik['R2']:.4f}")

df_benchmark = pd.DataFrame(hasil_benchmark)[["Model", "MAE (Rp)", "MAPE (%)", "R2"]].sort_values("R2", ascending=False)
display(df_benchmark)

## 7. Hyperparameter Tuning & Permutation Feature Importance

Berdasarkan hasil benchmark, **Random Forest Regressor** merupakan kandidat paling unggul dengan nilai $R^2$ tertinggi dan error terendah. Selanjutnya kita melakukan *hyperparameter tuning* menggunakan `RandomizedSearchCV` dengan 5-Fold Cross Validation.

In [ ]:
param_dist = {
    "regressor__regressor__n_estimators": [100, 150, 200],
    "regressor__regressor__max_depth": [15, 20, 25, 30],
    "regressor__regressor__min_samples_split": [2, 5],
    "regressor__regressor__min_samples_leaf": [1, 2]
}

base_rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", TransformedTargetRegressor(
        regressor=RandomForestRegressor(random_state=42, n_jobs=-1),
        func=np.log1p,
        inverse_func=np.expm1
    ))
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=base_rf_pipeline,
    param_distributions=param_dist,
    n_iter=8,
    scoring="neg_mean_absolute_error",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True
)

search.fit(X_train, y_train)

print("\nParameter Terbaik:")
print(search.best_params_)
print(f"Best CV MAE (Rp): {-search.best_score_:,.0f}")

### 7.1 Analisis Permutation Feature Importance
Menganalisis pengaruh masing-masing variabel prediktor terhadap tingkat error prediksi model final pada data testing.

In [ ]:
model_final = search.best_estimator_

perm = permutation_importance(
    model_final,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

df_importance = pd.DataFrame({
    "Fitur": X_test.columns,
    "Importance": perm.importances_mean,
    "Std": perm.importances_std
}).sort_values("Importance", ascending=False)

display(df_importance)

plt.figure(figsize=(9, 5))
plt.barh(df_importance["Fitur"][::-1], df_importance["Importance"][::-1], color="#2563eb")
plt.xlabel("Kenaikan Nilai Neg-MAE (Dampak terhadap Penurunan Error)")
plt.title("Permutation Feature Importance (Data Testing)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Evaluasi Model Akhir & Diagnostik Residual

Kita menguji model terbaik pada data testing yang belum pernah dilihat sebelumnya (*unseen test set*), menyandingkan perbandingannya terhadap baseline awal, serta memeriksa grafik sebaran prediksi vs aktual dan residual error.

In [ ]:
pred_final = model_final.predict(X_test)
hasil_final = evaluasi_model(y_test, pred_final)

print("=== HASIL EVALUASI MODEL AKHIR (TEST SET) ===")
print(f"MAE  : Rp {hasil_final['MAE (Rp)']:,.0f}")
print(f"MAPE : {hasil_final['MAPE (%)']:.2f}%")
print(f"R²   : {hasil_final['R2']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Scatter Plot Aktual vs Prediksi
axes[0].scatter(y_test / 1e9, pred_final / 1e9, alpha=0.3, color="#2563eb")
batas_min = min(y_test.min(), pred_final.min()) / 1e9
batas_max = max(y_test.max(), pred_final.max()) / 1e9
axes[0].plot([batas_min, batas_max], [batas_min, batas_max], "r--", linewidth=2)
axes[0].set_xlabel("Harga Aktual (Miliar Rp)", fontsize=11)
axes[0].set_ylabel("Harga Prediksi (Miliar Rp)", fontsize=11)
axes[0].set_title("Harga Aktual vs Prediksi Model (Garis Ideal Merah)", fontsize=13, fontweight="bold")

# 2. Residual Plot
residuals = (y_test - pred_final) / 1e9
axes[1].scatter(pred_final / 1e9, residuals, alpha=0.3, color="#059669")
axes[1].axhline(0, color="r", linestyle="--", linewidth=2)
axes[1].set_xlabel("Harga Prediksi (Miliar Rp)", fontsize=11)
axes[1].set_ylabel("Residual (Aktual - Prediksi) (Miliar Rp)", fontsize=11)
axes[1].set_title("Plot Residual vs Harga Prediksi", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

## 9. Model Persistence & Hilirisasi Deployment (Streamlit)

Setelah model terbukti unggul, kita mengekspor pipeline lengkap ke berkas `model_harga_rumah.joblib` sehingga dapat langsung dimanfaatkan oleh aplikasi antarmuka web Streamlit (`app.py`).

In [ ]:
# Menyimpan model ke disk
model_path = "model_harga_rumah.joblib"
joblib.dump(model_final, model_path)
print(f"Model berhasil disimpan ke berkas: {model_path}")

# Verifikasi simulasi pemuatan model seperti di Streamlit
model_loaded = joblib.load(model_path)

sample_input = pd.DataFrame([{
    "Kamar Tidur_num": 3,
    "Kamar Mandi_num": 2,
    "Garasi_num": 1,
    "Luas Tanah_num": 100.0,
    "Luas Bangunan_num": 80.0,
    "Kecamatan": "Cinere"
}])

estimasi_sample = float(model_loaded.predict(sample_input)[0])
print(f"Contoh Prediksi Properti Cinere (LT 100m², LB 80m², 3KT, 2KM): Rp {estimasi_sample:,.0f}")

### 9.1 Petunjuk Menjalankan Web App Streamlit
Untuk menjalankan aplikasi web interaktif berbasis model ini:
1. Pastikan dependensi di `requirements.txt` terpasang (`pip install -r requirements.txt`).
2. Jalankan perintah terminal:
   ```bash
   streamlit run app.py
   ```
3. Buka peramban di URL `http://localhost:8501`.